# Adding Structured Data to ChromaDB

In this notebook, we will demonstrate how to initialize a new local ChromaDB database and populate it with structured data from a CSV file. 

Unlike purely unstructured text (like raw PDF extracts), structured data often contains explicit fields (e.g., categories, dates, prices) that can be stored as **metadata**. In ChromaDB, we can embed the textual representation of the data while storing the structured attributes as metadata, which allows us to perform powerful hybrid queries (vector search + metadata filtering).

For this example, we will use the `medcare_provider_directory.csv` dataset, which contains information about medical providers.


In [1]:
import chromadb
import csv
import json

# Initialize a local ChromaDB client
# The database will be stored in the 'chroma_data' directory within the current folder
client = chromadb.PersistentClient(path="./chroma_data")

# Create or get a collection for our structured data
collection_name = "medcare_providers"
collection = client.get_or_create_collection(name=collection_name)

print(f"Collection '{collection_name}' is ready.")
print(f"Current document count: {collection.count()}")


Collection 'medcare_providers' is ready.
Current document count: 0


In [2]:
# Load the structured data from CSV
csv_path = "../1-medcare/data/medcare_provider_directory.csv"

providers = []
with open(csv_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        providers.append(row)

print(f"Loaded {len(providers)} providers.")
# Display the first provider as an example
print(json.dumps(providers[0], indent=2))


Loaded 50 providers.
{
  "Provider_ID": "MD-1001",
  "Full_Name": "Dr. Jones B.",
  "Specialty": "Family Medicine",
  "Affiliation": "Medcare Internal",
  "Facility_ID": "FAC-05",
  "Accepting_New_Patients": "Yes",
  "Language_Fluency": "English, Mandarin"
}


In [3]:
# Prepare data for insertion into ChromaDB

documents = []
metadatas = []
ids = []

for provider in providers:
    # The 'document' is the text that will be embedded and searched semantically
    document_text = f"{provider['Full_Name']} is a {provider['Specialty']} specialist affiliated with {provider['Affiliation']}. They speak {provider['Language_Fluency']}."
    
    # Metadata contains the structured fields for filtering
    metadata = {
        "specialty": provider["Specialty"],
        "facility_id": provider["Facility_ID"],
        "accepting_new_patients": provider["Accepting_New_Patients"] == "Yes",
        "affiliation": provider["Affiliation"]
    }
    
    documents.append(document_text)
    metadatas.append(metadata)
    ids.append(provider["Provider_ID"])

# Insert into the collection
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Successfully inserted {len(documents)} items into the collection.")
print(f"New document count: {collection.count()}")


C:\Users\yuval\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:09<00:00, 8.55MiB/s]


Successfully inserted 50 items into the collection.
New document count: 50


In [4]:
# Let's test our collection with a hybrid query!
# We want to find doctors who specialize in heart issues, but only those accepting new patients

results = collection.query(
    query_texts=["heart conditions, cardiovascular"],
    n_results=3,
    where={
        "accepting_new_patients": {"$eq": True}
    }
)

print("Query Results:")
for i in range(len(results['ids'][0])):
    print(f"ID: {results['ids'][0][i]}")
    print(f"Document: {results['documents'][0][i]}")
    print(f"Metadata: {results['metadatas'][0][i]}")
    print(f"Distance: {results['distances'][0][i]:.4f}")
    print("-" * 30)


Query Results:
ID: MD-1049
Document: Dr. Chen B. is a Cardiology specialist affiliated with Affiliate Partner. They speak English, None.
Metadata: {'facility_id': 'FAC-02', 'specialty': 'Cardiology', 'accepting_new_patients': True, 'affiliation': 'Affiliate Partner'}
Distance: 1.2744
------------------------------
ID: MD-1003
Document: Dr. Muller B. is a Cardiology specialist affiliated with Medcare Internal. They speak English, Mandarin.
Metadata: {'specialty': 'Cardiology', 'facility_id': 'FAC-01', 'accepting_new_patients': True, 'affiliation': 'Medcare Internal'}
Distance: 1.2919
------------------------------
ID: MD-1030
Document: Dr. Muller M. is a Cardiology specialist affiliated with Medcare Internal. They speak English, Mandarin.
Metadata: {'specialty': 'Cardiology', 'accepting_new_patients': True, 'facility_id': 'FAC-01', 'affiliation': 'Medcare Internal'}
Distance: 1.2923
------------------------------
